# Solver calibration figures (`sec:calibration`)

> **This notebook computes the validation panels; it does not write the paper's figures.**
> Every panel is drawn by a `make_calib_*.py` script at the house size (all text 10 pt
> printed, `figstyle.FONT`), from the plotted values in `data/figure_data/`, which this
> notebook computes from the validation simulations. The drawing cells below keep the old
> type sizes, so the first code cell pins `REPLACE_FIGURES = False`. To regenerate from the
> simulations, compute the arrays here, write them in the JSON schema of the matching file in
> `data/figure_data/`, then run that script.


Every panel is computed from the scored simulation NPZs and the published
anchor files in `flow3D_scripts/validation/anchors/`. Metric definitions come
from `flow3D_scripts/python/melt_metrics.py`, which is the single tested source
for fusion geometry, vapour depression and the domain-validity checks; nothing
is re-derived here.

Two rules the metrics enforce and this notebook inherits:

- **Depth kind.** Metallography reports the fusion boundary, in-situ X-ray the
  vapour depression. Each simulated quantity is scored only against the matching
  measurement type (`melt_metrics.DepthKindMismatch` refuses the other pairing).
- **Selective loading.** A refined-case NPZ holds tens of GB raw. Only the fields
  a panel needs are read; `{k: z[k] for k in z.files}` would materialise all of it.

Figures produced:

1. `calib_keyhole_ladder` -- simulated against measured depression depth, band shaded, cell counts annotated
2. `calib_transition_ladder` -- depth error and width error against power
3. `calib_spot_size` -- cross-sections at $r_0=60$ and $130\,\mu$m
4. `calib_aspect_ratio` -- measured against simulated aspect across the ladder
5. `calib_mesh_pair` -- the controlled 10 vs 5 um refinement


In [ ]:
import sys, pathlib, os, json
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import figstyle

SCRIPTS = pathlib.Path(os.environ.get('F3D_SCRIPTS_ROOT', pathlib.Path.cwd().resolve().parents[1] / 'flow3D_scripts'))
sys.path.insert(0, str(SCRIPTS / 'python'))
import melt_metrics as mm
import score_anchors as sa

figstyle.apply()
figstyle.REPLACE_FIGURES = False   # the make_calib_*.py scripts own these figures (see note above)
print('REPLACE_FIGURES =', figstyle.REPLACE_FIGURES)
print('metrics from   :', SCRIPTS / 'python' / 'melt_metrics.py')
BAND = mm.DEFAULT_BAND
print('acceptance band:', BAND)


In [ ]:
# Only the fields each metric needs. Loading every field would materialise
# tens of GB for a refined case (flow3D_scripts potential_bugs.md).
GEOM = ('Melt Region', 'Fraction Of Fluid', 'Temperature', 'x', 'y', 'z', 'time_s')

# The calibration runs were exported, ingested into AiiDA and then had their run
# directories reclaimed for disk, so most cases no longer live under runs/. The
# cache written by tools/fetch_calib_npz.py restores payload.npz + the manifest
# from the AiiDA node, in the layout below.
CACHE = pathlib.Path(os.environ.get('F3D_NPZ_CACHE',
                                    pathlib.Path.home() / '.flow3d' / 'npz_cache'))

def find_npz(case):
    hits = list(SCRIPTS.glob(f'runs/**/{case}/npz/{case}.npz'))
    if hits:
        return hits[0]
    cached = CACHE / case / 'npz' / f'{case}.npz'
    if cached.exists():
        return cached
    raise FileNotFoundError(
        f'{case}: not under runs/ and not in {CACHE}. '
        f'Restore it with: python3 tools/fetch_calib_npz.py {case}')

def load(case, fields=GEOM):
    with np.load(find_npz(case), allow_pickle=False) as z:
        return {k: z[k] for k in fields if k in z.files}

def manifest(case):
    return json.loads((find_npz(case).parent.parent / 'case_manifest.json').read_text())

def anchor_points(name, series=None):
    """Measured points through the scorer's own normaliser.

    The raw anchor files differ in schema (gan_points.json uses P/V) and
    gan carries depth_kind in a .meta.json sidecar, so reading them by hand
    is how a wrong-metric comparison gets made.
    """
    a = sa.load_anchor(SCRIPTS / 'validation' / 'anchors' / name)
    return sa.normalize_points(a, series=series)

def transverse_union(case):
    """Melt union and vapour cavity in the plane across the track at mid-x.

    This is the plane melt_metrics.fusion_geometry sections, so a panel drawn
    from it shows the same feature the scored number measures.

    The cavity is taken from the frame at which THIS plane is deepest, i.e. the
    moment the beam crosses it. Averaging over the steady window instead leaves
    almost no cavity: by then the beam is far downstream and the hole at mid-x
    has closed, which drew a contour hugging the surface.
    """
    d = load(case, ('Melt Region', 'Fraction Of Fluid', 'x', 'y', 'z', 'time_s'))
    melt = np.asarray(d['Melt Region']) > 0.5
    xc = melt.shape[1] // 2
    union = melt[:, xc, :, :].any(axis=0)                 # (y, z)
    y = (np.asarray(d['y']) - np.asarray(d['y']).mean()) * 1e6
    z = np.asarray(d['z']) * 1e6
    gas = None
    if 'Fraction Of Fluid' in d:
        G = 1.0 - np.asarray(d['Fraction Of Fluid'])[:, xc, :, :]   # (t, y, z)
        below = z < 0.0
        depth = [(np.where((g[:, below] > 0.5).any(axis=0))[0].min()
                  if (g[:, below] > 0.5).any() else G.shape[2]) for g in G]
        gas = G[int(np.argmin(depth))]
        meta = dict(frame=int(np.argmin(depth)), x_index=xc,
                    x_um=float(np.asarray(d['x'])[xc] * 1e6))
    return y, z, union, gas, meta

# One definition for the two mask colours, used for BOTH the pcolormesh and the
# legend patch. Deriving the patch from a colormap endpoint instead put a RED
# swatch next to a YELLOW cavity (caught 2026-08-15): a legend that misstates a
# colour is worse than no legend.
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
MELT_C = plt.get_cmap('Blues')(0.65)
CAV_C = '#F0E442'                      # Okabe-Ito yellow
CMAP_MELT = ListedColormap([MELT_C])
CMAP_CAV = ListedColormap([CAV_C])

def key_above(ax, handles, ncol=3):
    """Legend above the axes. Never inside: on these cross-sections every
    interior region is either pool, cavity or a measured-length line."""
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.01),
              ncol=ncol, frameon=False, fontsize=9, handlelength=1.5,
              columnspacing=1.0, handletextpad=0.4)


## 1. Keyhole ladder

In-situ X-ray depression depth against power at 0.40 m/s. Simulated depression
is scored against measured depression. Per-point cell counts are annotated so
the under-resolved 103 W condition stays visible.


In [ ]:
KEYHOLE = [('t2_103w', 103.0), ('t1_129w_tv3520_noplume', 129.0),
           ('t2_156w', 156.0), ('t2_206w', 206.0)]

gan_pts = anchor_points('gan_points.json')
assert {p['depth_kind'] for p in gan_pts} == {'depression'}, 'Gan must be depression'

def gan_depth(power, speed=0.4, tol=2.0):
    for p in gan_pts:
        if abs(float(p['power_w']) - power) < tol and abs(float(p['speed_m_s']) - speed) < 1e-6:
            return float(p['depth_um'])
    return np.nan

# `cells` is the cavity DEPTH in cell widths, not its width -- the cavity is 10
# to 16 cells across in every one of these runs, so quoting one number without
# saying which is read as the wrong one. It is also a TIME MEAN, like the depth
# it comes from: the cavity oscillates, and the scatter printed below is 0.2 to
# 1.5 cells. That is why the panels annotate a rounded integer; a decimal would
# claim precision the oscillation does not support.
rows = []
for case, P in KEYHOLE:
    d = load(case)
    off = manifest(case)['derived'].get('laser_off_time_s')
    dep = mm.depression(d, laser_off_s=off)
    cell = mm.cell_size_um(d)
    rows.append(dict(case=case, P=P, sim=dep.depth_um, meas=gan_depth(P),
                     cells=dep.depth_um / cell, cells_sd=dep.depth_sd_um / cell))
for r in rows:
    r['err'] = 100 * (r['sim'] - r['meas']) / r['meas']
    print(f"  {r['case']:24s} P={r['P']:5.0f} sim={r['sim']:7.1f} meas={r['meas']:7.1f} "
          f"err={r['err']:+6.1f}%  {r['cells']:4.1f} +- {r['cells_sd']:.1f} cells deep")

In [ ]:
fig, ax = figstyle.panel()
P = np.array([r['P'] for r in rows]); meas = np.array([r['meas'] for r in rows])
sim = np.array([r['sim'] for r in rows]); cells = np.array([r['cells'] for r in rows])

grid = np.linspace(P.min() * 0.95, P.max() * 1.05, 50)
fit = np.polyfit(P, meas, 1)
ax.fill_between(grid, np.polyval(fit, grid) * (1 - BAND), np.polyval(fit, grid) * (1 + BAND),
                color=figstyle.PALETTE[0], alpha=0.13, lw=0,
                label=f'$\\pm{BAND*100:.0f}\\%$ band')
ax.plot(P, meas, 'o-', color='k', ms=4, lw=1.0, label='measured')
# One simulated marker, not two. The panel used to split the points on
# mm.MIN_CELLS_PER_FEATURE, but every point here lands inside the band whether
# or not it clears that threshold, so the split implied a reliability
# difference the data does not show -- and the cell count is annotated on each
# point anyway, which is the honest form of the same information.
ax.plot(P, sim, 's', color=figstyle.PALETTE[0], ms=5, label='simulated')
for p, s, c in zip(P, sim, cells):
    ax.annotate(f'{c:.0f}', (p, s), textcoords='offset points', xytext=(0, 15),
                ha='center', fontsize=9, color='0.45')
ax.set_xlabel('Laser power (W)')
ax.set_ylabel(r'Depression depth ($\mu$m)')
ax.legend(frameon=False, loc='upper left', fontsize=10,
          title='numbers: cells across the depth', title_fontsize=9)
figstyle.save(fig, 'calib_keyhole_ladder')

## 1b. Speed ladder

The second scored axis of the same source: depression depth against scan
speed at two powers. Drawn as its own panel so the manuscript figure can
show both axes side by side and the prose can drop the numbers.


In [ ]:
SPEED = [('vsw3_p208_v06', 208.0, 0.6), ('vsw3_p208_v07', 208.0, 0.7),
         ('vsw3_p209_v09', 209.0, 0.9), ('vsw3_p209_v11', 209.0, 1.1),
         ('vsw32_p156_v06', 156.0, 0.6), ('vsw32_p156_v07', 156.0, 0.7),
         ('vsw32_p158_v09', 158.0, 0.9), ('vsw32_p157_v12', 157.0, 1.2)]

srows = []
for case, P, V in SPEED:
    d = load(case)
    off = manifest(case)['derived'].get('laser_off_time_s')
    dep = mm.depression(d, laser_off_s=off)
    cell = mm.cell_size_um(d)
    srows.append(dict(case=case, P=P, V=V, sim=dep.depth_um, meas=gan_depth(P, V),
                      cells=dep.depth_um / cell, cells_sd=dep.depth_sd_um / cell))
for r in srows:
    r['err'] = 100 * (r['sim'] - r['meas']) / r['meas']
    print(f"  {r['case']:18s} P={r['P']:5.0f} V={r['V']:.1f} sim={r['sim']:6.1f} "
          f"meas={r['meas']:6.1f} err={r['err']:+6.1f}%  "
          f"{r['cells']:4.1f} +- {r['cells_sd']:.1f} cells deep")

fig, ax = figstyle.panel()
for P_nom, colour, mark in ((208.0, figstyle.PALETTE[0], 's'),
                            (156.0, figstyle.PALETTE[3], 'D')):
    grp = sorted([r for r in srows if abs(r['P'] - P_nom) < 5], key=lambda r: r['V'])
    V = np.array([r['V'] for r in grp]); meas = np.array([r['meas'] for r in grp])
    sim = np.array([r['sim'] for r in grp]); cells = np.array([r['cells'] for r in grp])
    ax.fill_between(V, meas * (1 - BAND), meas * (1 + BAND), color=colour,
                    alpha=0.13, lw=0)
    ax.plot(V, meas, 'o-', color='k', ms=4, lw=1.0,
            label='measured' if P_nom == 208.0 else None)
    # No under-resolved split here either; see the power panel above.
    ax.plot(V, sim, mark, color=colour, ms=5, label='Simulated, ' + {208: '208\u2013209 W', 156: '156\u2013158 W'}.get(round(P_nom), f'{P_nom:.0f} W'))
    for v, s, c in zip(V, sim, cells):
        ax.annotate(f'{c:.0f}', (v, s), textcoords='offset points', xytext=(0, 13),
                    ha='center', fontsize=9, color='0.45')
ax.set_xlabel(r'Scan speed (m s$^{-1}$)')
ax.set_ylabel(r'Depression depth ($\mu$m)')
# Headroom for the annotation above the highest marker, which the axes would
# otherwise clip.
ax.set_ylim(top=ax.get_ylim()[1] * 1.16)
ax.legend(frameon=False, loc='upper right', fontsize=10,
          title='numbers: cells across the depth', title_fontsize=9)
figstyle.save(fig, 'calib_speed_ladder')

## 1c. What the three scored lengths are

A definition panel for the manuscript's metrics paragraph: one simulated
cross-section carrying fusion depth, fusion width and vapour-depression
depth, so the reader sees what each number measures before reading any
comparison. Drawn from the same plane and the same time-union rule the
scored metrics use.


In [ ]:
DEF_CASE = 't2_206w'
_d = load(DEF_CASE)
g = mm.fusion_geometry(_d)
dep = mm.depression(_d, laser_off_s=manifest(DEF_CASE)['derived'].get('laser_off_time_s'))
print(f'{DEF_CASE}: fusion {g.depth_um:.0f} x {g.width_um:.0f} um, '
      f'depression {dep.depth_um:.0f} um')

# (a) across the track, the plane the fusion metrics section
y, z, union, gas, DEF_META = transverse_union(DEF_CASE)
below = z < 0.0
gas_below = (gas > 0.5) & below[None, :] if gas is not None else None
rows = np.where(below)[0]
widths = [(np.where(union[:, k])[0].max() - np.where(union[:, k])[0].min() + 1)
          if union[:, k].any() else 0 for k in rows]
z_w = z[rows[int(np.argmax(widths))]]

fig, ax = figstyle.panel(frac=0.49)
ax.pcolormesh(y, z, np.where(union, 1.0, np.nan).T, cmap=CMAP_MELT,
              vmin=0, vmax=1, rasterized=True)
if gas_below is not None:
    ax.pcolormesh(y, z, np.where(gas_below, 1.0, np.nan).T, cmap=CMAP_CAV, vmin=0, vmax=1, rasterized=True)
ax.axhline(0, color='k', lw=0.8)
hw = g.width_um / 2.0
ann = dict(arrowprops=dict(arrowstyle='<->', color='k', lw=1.3))
ax.annotate('', xy=(-hw, z_w), xytext=(hw, z_w), **ann)
# the width arrow crosses the cavity, so the label needs a ground to read on
ax.text(0, z_w + 12, 'fusion width', ha='center', va='bottom', fontsize=11,
        bbox=dict(boxstyle='square,pad=0.15', fc='white', ec='none', alpha=0.85))
xd = -hw * 1.30
ax.annotate('', xy=(xd, 0), xytext=(xd, -g.depth_um), **ann)
ax.text(xd - 10, -g.depth_um / 2, 'fusion depth', ha='right', va='center',
        rotation=90, fontsize=11)
xp = hw * 1.30
ax.annotate('', xy=(xp, 0), xytext=(xp, -dep.depth_um), **ann)
ax.text(xp + 10, -dep.depth_um / 2, 'vapor\ndepression\ndepth', ha='left',
        va='center', fontsize=11)
# legend: the figure must be readable without the caption
import matplotlib.patches as mpatches
key_above(ax, [mpatches.Patch(facecolor=MELT_C, label='melted metal'),
               mpatches.Patch(facecolor=CAV_C, label='vapor cavity')], ncol=2)
ax.set_xlim(-hw * 2.6, hw * 2.8); ax.set_ylim(-g.depth_um * 1.2, 45)
ax.set_xlabel(r'$y$ from track centre ($\mu$m)')
ax.set_ylabel(r'$z$ ($\mu$m)'); ax.set_aspect('equal')
figstyle.save(fig, 'calib_metric_defs')


In [ ]:
# (b) along the track: the SAME instant and the SAME plane as (a), seen down the
# scan direction, so the reader can place the transverse cut in context (leading
# front, cavity, wake).
#
# Both of those used to be chosen independently here, and the two panels
# disagreed because of it. This cell picked the frame whose cavity is deepest
# ANYWHERE along x, while (a) picks the frame whose cavity is deepest IN ITS OWN
# PLANE, and it drew x relative to the beam at its own frame rather than at (a)'s
# plane. On t2_206w that was frame 164 (820 us, beam at x = 15 um) against (a)'s
# frame 181 (905 us, plane at x = 5 um), so the column the two panels share read
# as one continuous cavity -255..-5 um in (a) and as two broken runs, -205..-135
# and -95..-5, in (b). Both are taken from DEF_META now, so they cannot drift.
d2 = load(DEF_CASE, ('Melt Region', 'Fraction Of Fluid', 'x', 'y', 'z',
                     'time_s', 'Heat Absorption Rate'))
melt2 = np.asarray(d2['Melt Region']) > 0.5
F2 = np.asarray(d2['Fraction Of Fluid'])
xg, zg = np.asarray(d2['x']) * 1e6, np.asarray(d2['z']) * 1e6
jc = melt2.shape[2] // 2                      # track centreline in y
belowz = zg < 0.0
G2 = 1.0 - F2[:, :, jc, :]                    # (t, x, z) gas on the centreline
k = DEF_META['frame']                         # (a)'s frame, not our own
xb = DEF_META['x_um']                         # (a)'s plane, not the beam
union2 = melt2[:, :, jc, :].any(axis=0)       # time union, (x, z)
gas2 = (G2[k] > 0.5) & belowz[None, :]
_trk = mm.beam_track(d2)
_xbeam = float(_trk[k, 0]) * 1e6 if _trk is not None and np.isfinite(_trk[k, 0]) else np.nan
print(f"  frame {k} (= panel a), plane x = {xb:.0f} um, beam at x = {_xbeam:.0f} um")

# Same aspect as (a): the two are panels of one figure and must reduce to the
# same size on the page. At 0.55 this one printed visibly shorter than its twin.
fig, ax = figstyle.panel(frac=0.49, aspect=0.77)
ax.pcolormesh(xg - xb, zg, np.where(union2, 1.0, np.nan).T, cmap=CMAP_MELT,
              vmin=0, vmax=1, rasterized=True)
# CMAP_CAV, not 'autumn': the colormap endpoint and the legend patch must be the
# same constant or the key can disagree with the data (caught once already).
ax.pcolormesh(xg - xb, zg, np.where(gas2, 1.0, np.nan).T, cmap=CMAP_CAV,
              vmin=0, vmax=1, rasterized=True)
ax.axhline(0, color='k', lw=0.8)
ax.axvline(0, color='0.25', lw=1.2, ls='--')
# Both annotations sit in the clear band above the rim. Grey text laid over the
# blue mask was unreadable at print size, and the scan-direction label was
# printing on top of its own arrow.
ax.annotate('plane of (a)', xy=(0, 30), xytext=(-20, 58), fontsize=11,
            va='center', ha='right', color='0.25',
            arrowprops=dict(arrowstyle='-', color='0.25', lw=0.8))
ax.annotate('', xy=(205, 42), xytext=(110, 42),
            arrowprops=dict(arrowstyle='->', color='0.25', lw=1.4))
ax.text(157, 50, 'scan direction', fontsize=11, ha='center', va='bottom',
        color='0.25')
ax.set_xlim(-330, 220); ax.set_ylim(-g.depth_um * 1.2, 78); ax.set_aspect('equal')
ax.set_xlabel(r'$x$ from the plane of (a) ($\mu$m)'); ax.set_ylabel(r'$z$ ($\mu$m)')
# Same key as (a): a panel that shares a colour scheme must carry the scheme.
key_above(ax, [mpatches.Patch(facecolor=MELT_C, label='melted metal'),
               mpatches.Patch(facecolor=CAV_C, label='vapor cavity')], ncol=2)
figstyle.save(fig, 'calib_metric_defs_along')

# The shared column must now read identically in the two panels. Checked here
# rather than by eye, because the disagreement it catches is exactly the one
# that shipped.
_ia = DEF_META['x_index']
_col_b = (G2[k][_ia, :] > 0.5) & belowz
_col_a = ((1.0 - F2[k, _ia, jc, :]) > 0.5) & belowz
assert np.array_equal(_col_a, _col_b), 'panels (a) and (b) disagree on their shared column'
_zc = zg[_col_b]
print(f'  shared column agrees: gas from {_zc.max():.0f} to {_zc.min():.0f} um'
      if _col_b.any() else '  shared column agrees: no gas below the surface')

## 2. Transition ladder

Fusion depth and width error against power at $r_0=60\,\mu$m, 0.20 m/s.
The divergence of the two curves is the finding: width tracks the measurement
while depth departs from it.


In [ ]:
TRANS = [('spot60_p80_v020_std', 80.0), ('spot60_p90_v020_std', 90.0),
         ('spot60_p100_v020_std', 100.0), ('spot60_p110_v020_std', 110.0)]
# The 0.40 m/s series from the same source, plate and spot. It reaches across
# the keyhole threshold (cavity 17 -> 181 um over the five powers, against a
# 61 um beam radius), which is what lets the figure show the deficit collapsing
# with cavity depth instead of the prose reciting five percentages.
TRANS40 = [('spot60_p100_v040_std', 100.0), ('spot60_p150_v040_std', 150.0),
           ('spot60_p175_v040_std', 175.0), ('spot60_p200_v040_std', 200.0),
           ('spot60_p225_v040_std', 225.0)]
gal = anchor_points('galiegue2025_ti64_targets.json')
assert {p['depth_kind'] for p in gal} == {'fusion'}, 'Galiegue must be fusion'

def gal_point(power, r0_um=60.0, speed=0.20):
    for p in gal:
        if (p.get('r0_m') and abs(p['r0_m'] * 1e6 - r0_um) < 0.6
                and abs(p['power_w'] - power) < 0.5 and abs(p['speed_m_s'] - speed) < 1e-6):
            return float(p['depth_um']), float(p['width_um'])
    return np.nan, np.nan

def score_series(cases, speed):
    # fusion_geometry_auto with the beam field, not plain fusion_geometry: the
    # mid-x fallback mis-scores the 0.40 m/s cases whose deepest melt forms away
    # from the domain centre (same trap section 7 documents for CoCr 290 W).
    #
    # The vapour depression is scored here too, on the same loaded case, so the
    # conducted melt shell (fusion depth minus cavity depth) can be drawn as a
    # panel instead of quoted as two numbers in the text.
    rows = []
    for case, P in cases:
        d = load(case, GEOM + ('Heat Absorption Rate',))
        g = mm.fusion_geometry_auto(d); cell = mm.cell_size_um(d)
        dep = mm.depression(d, laser_off_s=manifest(case)['derived'].get('laser_off_time_s'))
        md_, mw_ = gal_point(P, speed=speed)
        rows.append(dict(P=P, d_err=100*(g.depth_um-md_)/md_, w_err=100*(g.width_um-mw_)/mw_,
                         d_cells=g.depth_um/cell, w_cells=g.width_um/cell,
                         sim_asp=g.depth_um/g.width_um, meas_asp=md_/mw_,
                         fus_um=g.depth_um, cav_um=dep.depth_um,
                         shell_um=g.depth_um - dep.depth_um))
    return rows

trows = score_series(TRANS, 0.20)
trows40 = score_series(TRANS40, 0.40)
for lbl, rows in (('0.20', trows), ('0.40', trows40)):
    for r in rows:
        print(f"  V={lbl}  P={r['P']:5.0f}  depth {r['d_err']:+6.1f}% ({r['d_cells']:4.1f} cells)  "
              f"width {r['w_err']:+6.1f}% ({r['w_cells']:4.1f} cells)  "
              f"shell {r['shell_um']:5.1f} um")
    _s = np.array([r['shell_um'] for r in rows])
    print(f"  V={lbl}  conducted melt shell {_s.mean():.0f} +- {_s.std():.0f} um "
          f"over {len(_s)} cases")

In [ ]:
fig, ax = figstyle.panel()
ax.axhspan(-BAND*100, BAND*100, color=figstyle.PALETTE[0], alpha=0.13, lw=0,
           label=f'$\\pm{BAND*100:.0f}\\%$ band')
ax.axhline(0, color='k', lw=0.6)
# Solid/filled = 0.20 m/s, dashed/open = 0.40 m/s, one colour per quantity, so
# the collapse of the depth error with power at 0.40 is read off the figure.
for rows, ls, mfc, tag in ((trows, '-', None, '0.20'), (trows40, '--', 'none', '0.40')):
    P_ = np.array([r['P'] for r in rows])
    ax.plot(P_, [r['d_err'] for r in rows], 'o', ls=ls, mfc=mfc,
            color=figstyle.PALETTE[5], ms=5,
            label=f'depth, {tag} ' + r'm s$^{-1}$')
    ax.plot(P_, [r['w_err'] for r in rows], 's', ls=ls, mfc=mfc,
            color=figstyle.PALETTE[2], ms=5,
            label=f'width, {tag} ' + r'm s$^{-1}$')
ax.set_xlabel('Laser power (W)')
ax.set_ylabel('Fusion-zone error (%)')
# Above the axes: the dashed depth line crosses the lower half, so any inside
# placement covers data.
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.01),
          ncol=3, fontsize=10, columnspacing=0.8, handletextpad=0.5,
          handlelength=1.7)
figstyle.save(fig, 'calib_transition_ladder')


## 3. Spot-size comparison

Simulated melt cross-sections at the same power and speed for $r_0=60$ and
$130\,\mu$m, in physical orientation. The larger spot needs almost no cavity,
and its depth lands inside the band while its width falls outside.


In [ ]:
# One PDF per panel: LaTeX assembles them with \subcaption so the text can cite
# (a) and (b). Both use panel(frac=0.49) so they reduce identically on the page.
import matplotlib.patches as mpatches
_KEY = lambda: [mpatches.Patch(facecolor=MELT_C, label='melted metal'),
                plt.Line2D([], [], color=figstyle.PALETTE[5], lw=2.2, ls='--', label='measured depth'),
                plt.Line2D([], [], color=figstyle.PALETTE[5], lw=1.2, ls=':', label='measured width')]

for _case, _r0, _tag in (('spot60_p100_v020_std', 60.0, 'r60'),
                         ('spot130_p100_v020_std', 130.0, 'r130')):
    fig, ax = figstyle.panel(frac=0.49, aspect=0.80)
    yy, zz, union, _, _ = transverse_union(_case)
    ax.pcolormesh(yy, zz, np.where(union, 1.0, np.nan).T, cmap=CMAP_MELT,
              vmin=0, vmax=1, rasterized=True)
    ax.axhline(0, color='k', lw=0.6)
    md_, mw_ = gal_point(100.0, r0_um=_r0)
    ax.axhline(-md_, color=figstyle.PALETTE[5], lw=2.2, ls='--')
    for s in (-1, 1):
        ax.axvline(s * mw_ / 2, color=figstyle.PALETTE[5], lw=1.2, ls=':')
    ax.set_xlabel(r'$y$ ($\mu$m)'); ax.set_ylabel(r'$z$ ($\mu$m)')
    ax.set_xlim(-185, 185); ax.set_ylim(-200, 45); ax.set_aspect('equal')
    key_above(ax, _KEY())
    print(f'  {_case}: measured depth {md_:.0f} um, width {mw_:.0f} um')
    figstyle.save(fig, f'calib_spot_size_{_tag}')


## 4. Aspect ratio across the ladder

Measured aspect climbs with power while the simulated pool stays flat, so the
simulated pool grows sideways where the measured pool grows downward.


In [ ]:
# (b) shape of the pool, carrying BOTH speeds with the same solid/open
# convention as the error panel: solid joined markers are 0.20 m/s, dashed and
# open are 0.40 m/s.
#
# The conducted melt shell had a panel here briefly and it is gone again: the
# numbers it drew (41 +- 2 um at 0.20, 34 +- 3 at 0.40, printed by the scoring
# cell above) are two values the text states in a line, and a panel of two flat
# series earns less than the space it took.
SPEEDS = ((trows, '-', None, '0.20'), (trows40, '--', 'none', '0.40'))

fig, ax = figstyle.panel()
for rows_, ls, mfc, tag in SPEEDS:
    P_ = np.array([r['P'] for r in rows_])
    ax.plot(P_, [r['meas_asp'] for r in rows_], 'o', ls=ls, mfc=mfc, color='k',
            ms=4, label=f'measured, {tag} ' + r'm s$^{-1}$')
    ax.plot(P_, [r['sim_asp'] for r in rows_], 's', ls=ls, mfc=mfc,
            color=figstyle.PALETTE[0], ms=4, label=f'simulated, {tag} ' + r'm s$^{-1}$')
ax.set_xlabel('Laser power (W)')
ax.set_ylabel('Melt pool aspect ratio')
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=2,
          fontsize=9, columnspacing=0.8, handletextpad=0.5, handlelength=1.7)
figstyle.save(fig, 'calib_aspect_ratio')

# THE 100 W POINT OF THE 0.40 m/s SERIES IS AN OUTLIER IN THE SOURCE, and it is
# the one place where the measured 0.40 curve sits ABOVE the 0.20 one, which is
# backwards for a scan speed. Read from Supplementary Data 1 of the source
# (41598_2025_27232_MOESM1_ESM.xlsx, sheet Supplementary_data), r0 = 60 um,
# zero defocus:
#   row  8   100 W, 0.40 m/s   depth 121.8 um   <- this point
#   row  9   125 W, 0.40 m/s   depth 111.1 um   <- MORE power, SHALLOWER pool
#   rows 10-17  150-400 W      depth linear in power, 1.131*P - 17.4 um
# That fit predicts 95.7 um at 100 W, so the tabulated 121.8 sits 27 % above its
# own series. Across speed at 100 W the same point breaks the ordering:
#   0.15 -> 178.1, 0.20 -> 148.0, 0.25 -> 98.7, 0.30 -> 85.9,
#   0.40 -> 121.8 (up again), 0.50 -> 38.1.
# Our transcription is exact (anchor 121.80 um = SI row 8), so this is the
# measurement and not a reading error.
#
# The authors never discuss it, but the paper states the two things that explain
# it. (i) Only the low-speed conditions were averaged: "Low-speed single-track
# experiments were observed at three different locations (0.5, 0.7, and 0.9 mm).
# The measured melt pool depths were averaged and plotted as a single data
# point. For the other conditions, melt pool depths were measured at only one
# location." So every 0.40 m/s point is ONE cross-section, unaveraged, while the
# 0.20 m/s series it is being compared against is a mean of three. (ii) Depth
# varies along a track: Supplementary Fig. S9, "around 10% relative change was
# measured at most". And this point sits exactly where they report their own
# diagnostic to be unstable -- "the onset of keyhole formation was found to be
# unstable under low-power conditions, with significant fluctuations in the AE
# signal amplitude within a single scan line" -- which is consistent with it
# being the only point in its series carrying no keyhole frequency at all.
#
# Left in the figure rather than dropped, because removing an inconvenient
# measured point needs a stated rule. But our -59 % there is not a like-for-like
# deficit: against the series trend it is -48 %.
_a20 = {r['P']: r['meas_asp'] for r in trows}
_a40 = {r['P']: r['meas_asp'] for r in trows40}
if 100.0 in _a20 and 100.0 in _a40:
    print(f"  measured aspect at 100 W: 0.20 m/s {_a20[100.0]:.3f}, "
          f"0.40 m/s {_a40[100.0]:.3f} "
          f"({'INVERTED, see the note in this cell' if _a40[100.0] > _a20[100.0] else 'ordered'})")

## 5. Controlled mesh refinement

The one pair differing in cell size alone, verified from the rendered solver
inputs. Halving the cell closes four points of a 53 % depth deficit and leaves
width unchanged, and at 5 um the depth is resolved by 13 cells.


In [ ]:
PAIR = [('spot60_p90_v020_std', r'10 $\mu$m', 'c10'), ('spot60_p90_mesh5', r'5 $\mu$m', 'c5')]
md_, mw_ = gal_point(90.0)
for _case, _lbl, _tag in PAIR:
    d = load(_case, ('Melt Region', 'x', 'y', 'z', 'time_s'))
    g = mm.fusion_geometry(d); cell = mm.cell_size_um(d)
    fig, ax = figstyle.panel(frac=0.49, aspect=0.80)
    yy, zz, union, _, _ = transverse_union(_case)
    ax.pcolormesh(yy, zz, np.where(union, 1.0, np.nan).T, cmap=CMAP_MELT,
              vmin=0, vmax=1, rasterized=True)
    ax.axhline(0, color='k', lw=0.6)
    ax.axhline(-md_, color=figstyle.PALETTE[5], lw=2.2, ls='--')
    for s in (-1, 1):
        ax.axvline(s * mw_ / 2, color=figstyle.PALETTE[5], lw=1.2, ls=':')
    ax.set_xlabel(r'$y$ ($\mu$m)'); ax.set_ylabel(r'$z$ ($\mu$m)')
    ax.set_xlim(-150, 150); ax.set_ylim(-200, 45); ax.set_aspect('equal')
    ax.annotate(f'{g.depth_um:.0f} ' + r'$\mu$m' + f' ({g.depth_um/cell:.0f} cells)',
                (0.03, 0.90), xycoords='axes fraction', fontsize=10, color='0.25')
    key_above(ax, _KEY())
    print(f'  {_lbl:6s} depth {g.depth_um:6.1f} um ({g.depth_um/cell:4.1f} cells)  '
          f'width {g.width_um:6.1f} um  err {100*(g.depth_um-md_)/md_:+.1f}%')
    figstyle.save(fig, f'calib_mesh_{_tag}')


## 6. In-situ radiography: depth and absorbed power through one scan

The panels above compare end states. This section compares the *history* of a
single scan, because the absorptance measurement and the radiographs are the same
shot: the radiograph file numbers are the `FrameNumber` column of the absorptance
record, and the two agree on frame time to 0.3 us.

Three panels: the depression depth measured frame by frame against the simulated
depth, the absorbed fraction from the same shot against the simulated one, and the
simulated gas boundary drawn onto a radiograph at a common micron scale.

Radiograph geometry is measured, not assumed. The frames sit at +6.50 deg and are
levelled; the substrate surface is the dark band visible ahead of the beam; and the
scale follows from the beam having to advance 70 um per stored frame at 0.7 m/s.
Fixing the beam track to a straight line leaves one free parameter, the column
where the laser switched on, and only the correct scale makes the frames agree on
it (109.12 +- 0.40 px at 2.01 um/px, against +-2.4 px at 1.96 and +-6.8 px at 2.19).


In [ ]:
# --------------------------------------------------------------------------
# In-situ radiography panels. Geometry constants are measured; see the section
# text and flow3D_scripts/validation for their derivation.
import os, glob, re
from PIL import Image
from scipy.ndimage import rotate as _rot, label as _label

PINO = pathlib.Path(os.environ.get('PINO_SIMULATIONS_ROOT',
                                   pathlib.Path.cwd().resolve().parents[1] / 'lpno'))
EXPDIR = PINO / 'data' / 'experimental'

TILT, SURF, UMPX, XON = 6.50, 92, 2.01, 109.12
TEXT_BOX = (slice(0, 30), slice(0, 200))
CASE, P_W, V_MS = 't2_nist195', 195.0, 0.7
# The 5 um twin differs from the published run in cell size only (prepin diff in
# the two input decks, 2026-08-06), so both meshes are drawn in every
# panel of this figure, as they are in the stationary-spot pair below.
CASE5 = 't2n195_mesh5'
LASER_OFF = manifest(CASE)['derived']['laser_off_time_s']

_num = lambda f: int(re.search(r'G1200(\d+)\.tif$', str(f)).group(1))
_tof = lambda f: (_num(f) - 98.634) * 20e-6

def _frame(f):
    raw = np.asarray(Image.open(f), float).copy()
    raw[TEXT_BOX] = np.median(raw)
    return _rot(raw, TILT, reshape=False, order=1, mode='nearest')

def _beam_px(t):
    return XON + (V_MS * t * 1e6) / UMPX

def _cavity_um(a, bpx):
    sub = a[SURF + 2:, max(int(bpx - 120 / UMPX), 0):int(bpx + 40 / UMPX)]
    lab, n = _label(sub > np.median(a) + 2.0 * a.std())
    if not n:
        return np.nan
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    big = int(np.argmax(sizes))
    if sizes[big] < 25:
        return np.nan
    return (np.where((lab == big).any(axis=1))[0].max() + 2) * UMPX

def _sim_series(case):
    """Depression depth and total deposited power per frame, laser-on only.

    Returns the two REDUCED series, not the fields: one field of the 5 um twin
    is 1 GB, and nothing downstream needs more than these.
    """
    zt = load(case, GEOM + ('Heat Absorption Rate',))
    k = int((zt['time_s'] <= LASER_OFF).sum())
    zt = {kk: (v[:k] if getattr(v, 'ndim', 0) >= 1 and v.shape[0] == len(zt['time_s'])
               else v) for kk, v in zt.items()}
    return (zt['time_s'], mm.depression_series(zt),
            zt['Heat Absorption Rate'].sum(axis=(1, 2, 3)))

_ts, _ser, _tot = _sim_series(CASE)
_ts5, _ser5, _tot5 = _sim_series(CASE5)

_frames = sorted(glob.glob(str(EXPDIR / 'xray' / '*.tif')))
_rows = []
for f in _frames:
    t = _tof(f)
    if not (0 < t <= _ts[-1]):
        continue
    _rows.append((t * 1e6, _cavity_um(_frame(f), _beam_px(t))))
_hist = np.array(_rows)
print(f'{len(_hist)} radiograph frames inside the simulated window')
for _c, _t_, _s_ in ((CASE, _ts, _ser), (CASE5, _ts5, _ser5)):
    print(f'  {_c:16s} cavity {np.nanmean(_s_[(_t_ > 0.4e-4) & (_t_ < 1.0e-4)]):5.1f} '
          f'-> {np.nanmean(_s_[(_t_ > 6e-4) & (_t_ < 8.5e-4)]):5.1f} um '
          f'(early -> settled)')

In [ ]:
# Shared treatment for all four panels of this figure.
#
# The shaded band is named in place rather than in the legend: it frees the
# legend to sit above the axes, where it cannot cover a trace, and it keeps the
# panels carrying the same three keys.
#
# Every spiky trace, measured or simulated, gets the same faint-raw + running-
# median treatment. Simulated depth is spiky because it is quantised to the cell;
# measured depth because the cavity genuinely oscillates and the segmentation
# adds to it; measured absorptance because it is sampled at 0.04 us. Smoothing
# some of them and not others made the same kind of scatter look like a property
# of whichever trace was left raw.
from scipy.ndimage import median_filter

def running_median(v, w=15):
    """Median over a centred window of w samples, edges held constant.

    A median, not a mean: it preserves the onset STEP that these panels are
    about, where a moving average would ramp it.
    """
    return median_filter(np.asarray(v, float), size=int(w), mode='nearest')

def faint_median(ax, x, y, colour, lw, label=None, w=15):
    ax.plot(x, y, '-', color=colour, lw=0.7, alpha=0.30)
    ax.plot(x, running_median(y, w), '-', color=colour, lw=lw, label=label)

def name_transient(ax):
    ax.text(200, 0.97, 'drilling transient', ha='center', va='top', fontsize=11,
            color='0.45', transform=ax.get_xaxis_transform())

def key_over(ax, ncol):
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=ncol,
              frameon=False, fontsize=12, handlelength=1.8, columnspacing=1.2)

# Same two colours and weights as the radiograph overlays and the spot pair, so
# a mesh keeps one identity across the whole section.
C10, C5 = figstyle.PALETTE[0], figstyle.PALETTE[1]
LW10, LW5 = 3.0, 1.6

fig, ax = figstyle.panel()
ax.axvspan(0, 400, color='0.85', alpha=0.5, lw=0)
ax.plot(_hist[:, 0], _hist[:, 1], 'o-', color='k', ms=5, lw=1.0, label='measured')
faint_median(ax, _ts * 1e6, _ser, C10, LW10, r'10 $\mu$m', w=9)
faint_median(ax, _ts5 * 1e6, _ser5, C5, LW5, r'5 $\mu$m', w=9)
ax.set_xlabel(r'Time ($\mu$s)')
ax.set_ylabel(r'Depression depth ($\mu$m)')
ax.set_ylim(0, np.nanmax(_hist[:, 1]) * 1.35)
name_transient(ax)
key_over(ax, 3)
figstyle.save(fig, 'calib_depth_history')

In [ ]:
import pandas as pd
_ab = pd.read_csv(EXPDIR / 'nist_ti64_scan_absorption.csv')
_t, _P, _A = (_ab['Time'].to_numpy() * 1e3, _ab['InputLaser'].to_numpy(),
              _ab['AbsoluteAbsorption'].to_numpy())
_on = _P > 50
_meas = np.where(_on, _A / np.maximum(_P, 1e-9), np.nan) * 100

# What is plotted for the simulation is the share of incident power RETAINED in
# the metal: shape from the deposition field, level from the hd3out thermal-energy
# audit. It is a lower bound on absorptance, and it is the same quantity panel (d)
# plots, so the two halves of the figure compare.
#
# This used to be derived here by hand. It now comes from
# melt_metrics.absorbed_fraction_series, which scales the trace so its mean over
# the calibration window EQUALS melt_metrics.absorbed_fraction on the same log --
# an identity that is a test in that repo, so the trace and the scalar cannot
# drift apart. The hand-rolled version is what let a bad evaporative term reach
# this figure in the first place.
#
# No evaporative correction is added back. `total fluid mass` is a NET
# phase-change balance, evaporation minus recondensation (every deck renders
# rsize_cond=1 and acondens=1.0), so it is not a record of what left the pool.
# See melt_metrics.gross_absorptance, which now refuses such a series outright.
fig, ax = figstyle.panel()
ax.axvspan(0, 400, color='0.85', alpha=0.5, lw=0)
_w = _on & (_t * 1e3 <= _ts[-1] * 1e6)
# 0.04 us sampling, so w=1001 is a 40 us window -- the same window in TIME as
# w=9 on the ~4 us simulated frames, which is what makes the traces comparable.
faint_median(ax, _t[_w] * 1e3, _meas[_w], 'k', 1.6, 'measured', w=1001)
for _c, _ts_, _tot_, _col, _lw, _lab in (
        (CASE, _ts, _tot, C10, LW10, r'10 $\mu$m'),
        (CASE5, _ts5, _tot5, C5, LW5, r'5 $\mu$m')):
    _hd = find_npz(_c).parent.parent / f'hd3out.{_c}'
    _s = mm.absorbed_fraction_series(_ts_, _tot_, _hd, power_w=P_W,
                                     t0=2e-4, t1=8e-4)
    _r = 100 * _s.retained
    faint_median(ax, _ts_ * 1e6, _r, _col, _lw, _lab, w=9)
    _sm_ = running_median(_r, 9)
    print(f'  {_c:16s} cold {_sm_[1]:5.1f} %  transient peak {np.nanmax(_sm_):5.1f} %  '
          f'settled {np.nanmean(_sm_[(_ts_ > 6e-4) & (_ts_ < 8.5e-4)]):5.1f} %  '
          f'(window mean {_s.window_mean * 100:.1f} % = absorbed_fraction)')
_msm = running_median(_meas[_w], 1001)
print(f'  measured         cold {_msm[:200].mean():5.1f} %  '
      f'transient peak {np.nanmax(_msm):5.1f} %  settled {_msm[-2000:].mean():5.1f} %')
ax.set_xlabel(r'Time ($\mu$s)')
ax.set_ylabel('Absorbed fraction (%)')
ax.set_ylim(20, 80)
name_transient(ax)
key_over(ax, 3)
figstyle.save(fig, 'calib_absorptance_history')

In [ ]:
# Both meshes drawn on the same radiograph. The 5 um twin (t2n195_mesh5, pk
# 33553) differs from the published run in cell size only -- verified by prepin
# diff of the two input decks, 2026-08-06 -- so the gap between the two
# contours is the resolution sensitivity of the cavity and nothing else.
CASE5 = 't2n195_mesh5'
_x0 = manifest(CASE)['input']['laser']['start_point_m'][0]

def _centreline(case):
    """Gas fraction on the track centreline, plus the coordinates to draw it.

    Sliced at load time: the 5 um NPZ holds ~1 GB of Fraction Of Fluid and only
    one j-plane of it is ever plotted.
    """
    d = load(case, ('Fraction Of Fluid', 'x', 'z', 'time_s'))
    F = np.asarray(d['Fraction Of Fluid'])
    return dict(gas=1.0 - F[:, :, F.shape[2] // 2, :],      # (t, x, z)
                x=np.asarray(d['x']), z=np.asarray(d['z']),
                t=np.asarray(d['time_s']))

_cl = {c: _centreline(c) for c in (CASE, CASE5)}
for c, v in _cl.items():
    print(f"  {c:16s} {v['gas'].shape[1]:3d} x {v['gas'].shape[2]:3d} cells in x,z")

# Okabe-Ito blue/orange: distinct on a grayscale radiograph, safe for
# colour-vision deficiency. Where the two meshes agree the contours coincide
# exactly, so the wider line goes underneath or only one of them is visible.
def scale_bar(ax, length_um=100, pad=0.055, colour='w'):
    """Visual length reference, drawn after the limits are fixed.

    The axes already carry micron ticks, but a bar gives the physical size at a
    glance and survives the figure being resized or reproduced in a talk.
    """
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    xe = x1 - pad * (x1 - x0); xs = xe - length_um
    y = y0 + pad * (y1 - y0)
    ax.plot([xs, xe], [y, y], color=colour, lw=3.0, solid_capstyle='butt',
            zorder=5)
    ax.text((xs + xe) / 2, y + 0.035 * (y1 - y0), f'{length_um:g} ' + r'$\mu$m',
            color=colour, ha='center', va='bottom', fontsize=11, zorder=5)
    # Assert the drawn bar really is length_um, on every render rather than once
    # by hand. The check is a RATIO of two transData measurements taken at the
    # same moment, so it is immune to the transform being pre-draw and stale
    # (measured 9 % off before constrained_layout settles the axes position).
    # It catches what actually goes wrong: placement in axes fractions, a log
    # axis, or a round line cap extending the bar past its data span.
    tr = ax.transData
    span_px = (tr.transform((xe, y)) - tr.transform((xs, y)))[0]
    unit_px = (tr.transform((x0 + 1.0, y)) - tr.transform((x0, y)))[0]
    got = span_px / unit_px
    assert abs(got - length_um) < 1e-6, (
        f'scale bar is {got:.6f} um on the axes, not {length_um}')

C10, C5 = figstyle.PALETTE[0], figstyle.PALETTE[1]
LW10, LW5 = 3.0, 1.6

def _draw(ax, case, colour, lw, t_target):
    v = _cl[case]
    k = int(np.argmin(np.abs(v['t'] - t_target)))
    beam = _x0 + V_MS * v['t'][k]
    ax.contour((v['x'] - beam) * 1e6, v['z'] * 1e6, v['gas'][k].T,
               levels=[0.5], colors=[colour], linewidths=lw)

# Flat-field reference from the five pre-laser frames (t < 0), the same
# treatment as the stationary-spot panels: each frame is divided by the mean
# undisturbed frame, which removes the detector's fixed pattern.
_ref_scan = np.mean([_frame(g) for g in _frames if _tof(g) < 0], axis=0)

for _target, _tag in ((127e-6, 'transient'), (627e-6, 'settled')):
    _f = min((f for f in _frames if 0 < _tof(f) <= _ts[-1]),
             key=lambda f: abs(_tof(f) - _target))
    _t2 = _tof(_f); _a = _frame(_f); _bp = _beam_px(_t2)
    _flat = _a / np.maximum(_ref_scan, 1e-6)
    fig, ax = figstyle.panel(frac=0.48, aspect=0.55)
    ext = [(0 - _bp) * UMPX, (_a.shape[1] - _bp) * UMPX,
           (_a.shape[0] - SURF) * -UMPX, (0 - SURF) * -UMPX]
    ax.imshow(_flat, cmap='gray', aspect='equal', extent=ext, origin='upper',
              vmin=np.percentile(_flat, 2), vmax=np.percentile(_flat, 99.5))
    _draw(ax, CASE, C10, LW10, _t2)
    _draw(ax, CASE5, C5, LW5, _t2)
    ax.axhline(0, color='0.3', lw=0.6, ls=':')
    # -305 not -450: the 127 us frame's left edge sits at -307 um from the
    # beam, so the old limit left a white void and half-hid the time stamp.
    # The 627 us frame reaches -658, so both panels stay fully covered.
    ax.set_xlim(-305, 175); ax.set_ylim(-170, 80)
    ax.set_xlabel(r'$x$ from beam ($\mu$m)')
    ax.set_ylabel(r'$z$ ($\mu$m)')
    scale_bar(ax)
    ax.text(0.02, 0.06, f'$t$ = {_t2*1e6:.0f} ' + r'$\mu$s',
            transform=ax.transAxes, color='w', fontsize=13, va='bottom')
    ax.legend(handles=[plt.Line2D([], [], color=C10, lw=LW10,
                                  label=r'10 $\mu$m'),
                       plt.Line2D([], [], color=C5, lw=LW5,
                                  label=r'5 $\mu$m')],
              loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=2,
              frameon=False, fontsize=13, handlelength=1.6, columnspacing=1.2)
    figstyle.save(fig, f'calib_xray_{_tag}')
    print(f'  {_tag}: t = {_t2*1e6:.0f} us')

# Depth of each contour at that frame, the numbers the caption quotes.
def _cavity_depth(case, t_target):
    v = _cl[case]
    k = int(np.argmin(np.abs(v['t'] - t_target)))
    below = v['z'] < 0.0
    cols = (v['gas'][k][:, below] > 0.5).any(axis=0)   # z-columns holding gas
    if not cols.any():
        return np.nan
    return -v['z'][below][np.where(cols)[0].min()] * 1e6

for _target in (127e-6, 627e-6):
    d10_, d5_ = (_cavity_depth(c, _target) for c in (CASE, CASE5))
    print(f'  t = {_target*1e6:.0f} us: 10 um {d10_:.0f} um, 5 um {d5_:.0f} um deep')


## 6c. Stationary spot: the same two diagnostics, where resolution decides

The second configuration of the same NIST record (`mds2-2525`), a 101.9 W pulse
held on one spot for 2.008 ms. It is the case that separates a resolution limit
from a missing physical term, because the long dwell lets a cavity narrow and
deepen until the mesh either follows it or does not.

Three panels, all joining figures that already exist rather than forming a new
one (a new case is a new panel):

- `calib_spot_depth` and `calib_spot_abs` join the scanned depth/absorptance
  pair, so one figure carries both configurations under the same two
  diagnostics.
- `calib_spot_overlay` joins the radiograph figure as a third panel.

Geometry constants are fitted in `lpno/src/notebooks/spot_radiographs.py`
and differ from the scanned ones; the frame-time map comes from the
`FrameNumber` column of `Spot_Calibrated_Absorption.csv`, not a nominal 20 us.
The **raw** frames are used: `processed_captioned` is a mirrored crop with
burnt-in timestamps and only 148 of the 451 frames.


In [ ]:
SPOT = EXPDIR / 'spot'
S10, S5 = 'spot_nist102_2ms_sm_10um', 'spot_nist102_2ms_sm_5um'
SPOT_P_W, SPOT_PULSE_US = 101.9, 2008.4

# Fitted in lpno/src/notebooks/spot_radiographs.py; all differ from
# the scanned constants above except the micron scale, which is transferred.
SP_TILT, SP_UMPX, SP_SURF, SP_XBEAM = -6.164, 2.01, 92.7, 285.4
# From the CSV FrameNumber column. The interval is measured, not nominal: using
# 20.0 us drifts ~2 frames by the end of the pulse.
sp_num = lambda f: int(re.search(r'G1200(\d+)\.tif$', str(f)).group(1))
sp_tof = lambda n: -52.48 + (n - 80) * 19.987

sp_meas = np.array(json.loads(
    (SPOT / 'depth_series_thresh1.10.json').read_text())['series'])   # frame, us, um
sp_frames = sorted(glob.glob(str(SPOT / 'raw' / '*' / '*.tif')))
print(f'{len(sp_frames)} raw spot frames, measured peak '
      f'{sp_meas[:, 2].max():.0f} um at {sp_meas[sp_meas[:, 2].argmax(), 1]:.0f} us')

# running_median / faint_median come from the depth-history cell above, so all
# four panels of the figure smooth the same way.
sp_depth = {}
for _c in (S10, S5):
    _d = load(_c, ('Melt Region', 'Fraction Of Fluid', 'Temperature',
                   'x', 'y', 'z', 'time_s'))
    sp_depth[_c] = (np.asarray(_d['time_s']) * 1e6, mm.depression_series(_d))
    _t, _v = sp_depth[_c]
    _hit = np.where(_v > 60.0)[0]
    print(f'  {_c:30s} peak {np.nanmax(_v):5.0f} um  '
          f'onset(>60um) {(f"{_t[_hit[0]]:.0f} us" if len(_hit) else "never")}')

In [ ]:
SP_C10, SP_C5 = C10, C5                       # same identity as (a, b)
SP_LW10, SP_LW5 = LW10, LW5

def sp_key(ax):
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=3,
              frameon=False, fontsize=12, handlelength=1.6, columnspacing=1.0)

# (c) depression depth through the pulse.
# The measured trace gets the same faint-raw + median treatment as the simulated
# ones. w=5 (100 us at 20 us frame spacing), not the w=15 used on the ~4.5 us
# simulated frames: matching the window in SAMPLES here would be a 300 us median
# and would flatten the measured peak the text quotes. The raw trace stays
# visible underneath, so the peak is still there to read.
fig, ax = figstyle.panel(frac=0.49, aspect=0.77)
faint_median(ax, sp_meas[:, 1], sp_meas[:, 2], 'k', 1.5, 'measured', w=5)
for _c, _col, _lw, _lab in ((S10, SP_C10, SP_LW10, r'10 $\mu$m'),
                            (S5, SP_C5, SP_LW5, r'5 $\mu$m')):
    _t, _v = sp_depth[_c]
    faint_median(ax, _t, _v, _col, _lw, _lab)
ax.set_xlim(0, SPOT_PULSE_US); ax.set_ylim(0, 215)
ax.set_xlabel(r'Time ($\mu$s)'); ax.set_ylabel(r'Depression depth ($\mu$m)')
sp_key(ax)
figstyle.save(fig, 'calib_spot_depth')

# (d) absorbed fraction through the same pulse, the independent channel.
# Same quantity and the same library call as panel (b): the share RETAINED in the
# metal, a lower bound, not normalised to the measurement. What the panel
# supports is WHERE THE STEP IS, not the height after it.
#
# The window is 0-670 us and that is not arbitrary. This audit prints only every
# ~220 us, and the t=0 entry has no deposition yet so it drops out; the window
# that stood here before, 100-400 us, therefore held exactly ONE usable sample,
# and a median over one point returns it without complaining. 0-670 us holds
# three and is still pre-onset for both meshes, since the 5 um keyhole opens at
# 675 us -- 15 us of margin, which is why the bound is 670 and not a round 700.
# absorbed_fraction_series returns NaN rather than calibrating on fewer than two.
_ab = pd.read_csv(SPOT / 'Spot_Calibrated_Absorption.csv', low_memory=False)
_mt, _mP, _mA = (_ab['Time'].to_numpy() * 1e6, _ab['InputLaser'].to_numpy(),
                 _ab['AbsoluteAbsorption'].to_numpy())
_won = (_mP > 50) & (_mt >= 0) & (_mt <= SPOT_PULSE_US)

fig, ax = figstyle.panel(frac=0.49, aspect=0.77)
# 0.04 us sampling: w=1001 is a 40 us window, the same window in TIME as w=9 on
# the simulated frames.
faint_median(ax, _mt[_won], 100 * _mA[_won] / _mP[_won], 'k', 1.5, 'measured', w=1001)
for _c, _col, _lw, _lab in ((S10, SP_C10, SP_LW10, r'10 $\mu$m'),
                            (S5, SP_C5, SP_LW5, r'5 $\mu$m')):
    _d = load(_c, ('Heat Absorption Rate', 'time_s'))
    _t = np.asarray(_d['time_s'])
    _tot = np.asarray(_d['Heat Absorption Rate']).sum(axis=(1, 2, 3))
    _hd = find_npz(_c).parent.parent / f'hd3out.{_c}'
    _s = mm.absorbed_fraction_series(_t, _tot, _hd, power_w=SPOT_P_W,
                                     t0=0.0, t1=6.7e-4)
    assert np.isfinite(_s.scale), f'{_c}: too few audit samples to calibrate on'
    _frac = 100 * _s.retained
    faint_median(ax, _t * 1e6, _frac, _col, _lw, _lab, w=9)
    _pre = (_t > 1.0e-4) & (_t < 4.0e-4); _late = (_t > 1.2e-3) & (_t < 1.9e-3)
    print(f'  {_c:30s} pre-onset {np.nanmean(_frac[_pre]):5.1f} %  '
          f'late {np.nanmean(_frac[_late]):5.1f} %  '
          f'(window mean {_s.window_mean * 100:.1f} % = absorbed_fraction)')
ax.set_xlim(0, SPOT_PULSE_US); ax.set_ylim(20, 80)
ax.set_xlabel(r'Time ($\mu$s)'); ax.set_ylabel('Absorbed fraction (%)')
sp_key(ax)
figstyle.save(fig, 'calib_spot_abs')

In [ ]:
# (c, d) of the radiograph figure: both simulated boundaries on the spot frames,
# flat-fielded against the pre-laser frames the way the depth extraction is, so
# the measured cavity is visible against the detector's own structure.
#
# Two times, chosen to show the finding rather than to look nice. 847 us is the
# last frame BEFORE the measured onset (867 us) and well after the simulated one
# (675 us): the refined run is already 68 um down while the radiograph still
# reads 15 um. 1407 us is past both, where the depths agree. That pair is the
# "right depth, too early" result made visible. Same frac as (a, b), so all four
# panels of the figure reduce by the same factor on the page.
#
# These frames also show the melt halo around the cavity, which the scanned-track
# frames in (a, b) do not, so the simulated fusion boundary is drawn here as a
# dashed line of the same colour. It is the melt region UNIONED with the
# sub-surface cavity: melt alone is a shell, and contouring a shell draws two
# nested lines where the reader wants one outer boundary.
_ref_spot = np.mean([_rot(np.asarray(Image.open(g), float), SP_TILT,
                          reshape=False, order=1, mode='nearest')
                     for g in sp_frames if 20 <= sp_num(g) <= 79], axis=0)

def sp_planes(case):
    d = load(case, ('Fraction Of Fluid', 'Melt Region', 'x', 'z', 'time_s'))
    F = np.asarray(d['Fraction Of Fluid']); M = np.asarray(d['Melt Region'])
    j = F.shape[2] // 2
    return dict(gas=1.0 - F[:, :, j, :], melt=M[:, :, j, :],
                x=np.asarray(d['x']), z=np.asarray(d['z']),
                t=np.asarray(d['time_s']) * 1e6)

_spcl = {c: sp_planes(c) for c in (S10, S5)}

for _target, _tag in ((847.0, 'early'), (1407.0, 'late')):
    _f = min(sp_frames, key=lambda f: abs(sp_tof(sp_num(f)) - _target))
    _t2 = sp_tof(sp_num(_f))
    _a = _rot(np.asarray(Image.open(_f), float), SP_TILT, reshape=False,
              order=1, mode='nearest')
    _flat = _a / np.maximum(_ref_spot, 1e-6)
    fig, ax = figstyle.panel(frac=0.48, aspect=0.85)
    _ext = [(0 - SP_XBEAM) * SP_UMPX, (_a.shape[1] - SP_XBEAM) * SP_UMPX,
            (_a.shape[0] - SP_SURF) * -SP_UMPX, (0 - SP_SURF) * -SP_UMPX]
    ax.imshow(_flat, cmap='gray', aspect='equal', extent=_ext, origin='upper',
              vmin=np.percentile(_flat, 2), vmax=np.percentile(_flat, 99.5))
    for _c, _col, _lw in ((S10, SP_C10, SP_LW10), (S5, SP_C5, SP_LW5)):
        _v = _spcl[_c]
        _k = int(np.argmin(np.abs(_v['t'] - _t2)))
        _X = (_v['x'] - _v['x'].mean()) * 1e6
        _Z = _v['z'] * 1e6
        _fus = np.maximum(_v['melt'][_k], _v['gas'][_k] * (_Z < 0))
        ax.contour(_X, _Z, _fus.T, levels=[0.5], colors=[_col],
                   linewidths=_lw * 0.6, linestyles='--')
        ax.contour(_X, _Z, _v['gas'][_k].T, levels=[0.5], colors=[_col],
                   linewidths=_lw)
    ax.axhline(0, color='0.3', lw=0.6, ls=':')
    ax.set_xlim(-190, 190); ax.set_ylim(-250, 80)
    ax.set_xlabel(r'$x$ ($\mu$m)'); ax.set_ylabel(r'$z$ ($\mu$m)')
    scale_bar(ax)
    ax.text(0.03, 0.04, f'$t$ = {_t2:.0f} ' + r'$\mu$s', transform=ax.transAxes,
            color='w', fontsize=13, va='bottom')
    ax.legend(handles=[plt.Line2D([], [], color=SP_C10, lw=SP_LW10,
                                  label=r'10 $\mu$m'),
                       plt.Line2D([], [], color=SP_C5, lw=SP_LW5,
                                  label=r'5 $\mu$m'),
                       plt.Line2D([], [], color='0.35', lw=1.6, ls='--',
                                  label='fusion boundary')],
              loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=3,
              frameon=False, fontsize=11, handlelength=1.6, columnspacing=1.0)
    figstyle.save(fig, f'calib_spot_overlay_{_tag}')
    # depth of each simulated cavity and its fusion boundary at this frame
    for _c in (S10, S5):
        _v = _spcl[_c]; _k = int(np.argmin(np.abs(_v['t'] - _t2)))
        _below = _v['z'] < 0
        _cav = (_v['gas'][_k][:, _below] > 0.5).any(axis=0)
        _mlt = (_v['melt'][_k][:, _below] > 0.5).any(axis=0)
        _dep = lambda m: (-_v['z'][_below][np.where(m)[0].min()] * 1e6
                          if m.any() else np.nan)
        print(f'  {_tag} {_c:30s} cavity {_dep(_cav):4.0f}  fusion {_dep(_mlt):4.0f} um')
    _mi = int(np.argmin(np.abs(sp_meas[:, 1] - _t2)))
    print(f'  {_tag}: t = {_t2:.0f} us (frame {sp_num(_f)})  '
          f'measured cavity {sp_meas[_mi, 2]:.0f} um')

## 7. Agreement across materials

Measured against simulated length for every material we scored, so the
per-material result is one panel rather than a paragraph of signed errors.
Conditions come from each source's own anchor file through the scorer's
normaliser, and cases that fail `domain_validity` are dropped rather than
scored (the 456 W spot weld melts to the bottom of its domain).


In [ ]:
# ── Campaign summary: every scored condition in one place ─────────────────────
# Replaces the per-material parity pair. The point is coverage: a reader should be
# able to see WHERE the solver is accurate and draw their own conclusion, without
# the prose reciting percentages.
#
# Two things this fixes over the old materials figure:
#   1. It carried only the four 0.20 m/s Ti-6Al-4V cases, which are the worst
#      ones, so Ti-6Al-4V looked far weaker than the campaign shows.
#   2. It had no vapour-depression points at all, so the observable that actually
#      drives the deficit was absent.
#
# fusion_geometry_auto needs `Heat Absorption Rate` to find the beam; without it
# the mid-x fallback moved the CoCr 290 W case by a cell.
GEOM_BEAM = GEOM + ('Heat Absorption Rate',)
MARK = {'Ti-6Al-4V': 'o', 'CoCr': 's', '304L': 'D', 'IN718': '^'}
MCOL = {'Ti-6Al-4V': figstyle.PALETTE[0], 'CoCr': figstyle.PALETTE[1],
        '304L': figstyle.PALETTE[2], 'IN718': figstyle.PALETTE[3]}

# FUSION families: measured fusion depth and width (metallography).
FUSION = [
    ('Ti-6Al-4V', 'galiegue2025_ti64_targets.json', None,
     [('spot60_p80_v020_std', 80., .20, 60.), ('spot60_p90_v020_std', 90., .20, 60.),
      ('spot60_p100_v020_std', 100., .20, 60.), ('spot60_p110_v020_std', 110., .20, 60.),
      ('spot60_p100_v040_std', 100., .40, 60.), ('spot60_p150_v040_std', 150., .40, 60.),
      ('spot60_p175_v040_std', 175., .40, 60.), ('spot60_p200_v040_std', 200., .40, 60.),
      ('spot60_p225_v040_std', 225., .40, 60.), ('spot130_p100_v020_std', 100., .20, 130.)]),
    ('CoCr', 'appsci_cocr_targets.json', None,
     [('cocr_p120_v04', 121., .40, 50.), ('cocr_p200_v095', 200., .95, 50.),
      ('cocr_p290_v095', 290., .95, 50.)]),
    ('304L', 'hawk304l_targets.json', 'wrought_304L_AR_GR',
     [('s304t2_p118', 118., 0., 151.5), ('s304t2_p255', 255., 0., 151.5),
      ('s304t2_p355', 355., 0., 151.5), ('s304t2_p456', 456., 0., 151.5)]),
    ('IN718', 'amb718_targets.json', None,
     [('p3a_in718_noplume', 285., .96, 33.5), ('rep3_in718_p245', 245., .96, 33.5),
      ('rep3_in718_p325', 325., .96, 33.5), ('rep3_in718_v080', 285., .80, 33.5),
      ('rep3_in718_v120', 285., 1.20, 33.5), ('rep3_in718_s082', 285., .96, 41.0),
      ('rep4_in718_s049_blind', 285., .96, 24.5)]),
]
# DEPRESSION family: measured vapour-depression depth (in-situ X-ray, Gan). All
# Ti-6Al-4V. Power series at 0.40 m/s plus the speed series -- 12 conditions, the
# same twelve plotted in the keyhole figure.
DEPRESSION = [('t2_103w', 103., .40), ('t1_129w_tv3520_noplume', 129., .40),
              ('t2_156w', 156., .40), ('t2_206w', 206., .40),
              ('vsw3_p208_v06', 208., .60), ('vsw3_p208_v07', 208., .70),
              ('vsw3_p209_v09', 209., .90), ('vsw3_p209_v11', 209., 1.10),
              ('vsw32_p156_v06', 156., .60), ('vsw32_p156_v07', 156., .70),
              ('vsw32_p158_v09', 158., .90), ('vsw32_p157_v12', 157., 1.20)]

def amb718_targets():
    raw = json.loads((SCRIPTS / 'validation' / 'anchors' / 'amb718_targets.json').read_text())
    out = []
    for key, v in raw.items():
        P, V_mm, d4 = (float(t) for t in key.strip('()').split(','))
        out.append(dict(power_w=P, speed_m_s=V_mm / 1000., r0_m=d4 / 2 * 1e-6,
                        depth_um=v['depth_um'], width_um=v['width_um']))
    return out

def target(pts, P, V, r0_um):
    for q in pts:
        if (abs(q['power_w'] - P) <= max(1., .02 * P) and abs(q['speed_m_s'] - V) < 1e-3
                and (q.get('r0_m') is None or abs(q['r0_m'] * 1e6 - r0_um) < 1.)):
            return float(q['depth_um']), float(q.get('width_um') or np.nan)
    return np.nan, np.nan

def crown_um(d):
    """Height the bead stands above the original surface, per case.

    The sources do not state whether their depths run from the plate surface or
    from the top of the resolidified crown. Ours run from the surface, so a
    crown-referenced convention would add this. It is NOT the constant ~20 um the
    text used to assume: measured here it runs 5 um at 80 W to 45 um at 225 W.
    """
    melt = np.asarray(d['Melt Region']) > 0.5
    z = np.asarray(d['z']); above = z > 0.
    if not above.any():
        return 0.0
    col = melt[:, :, :, above].any(axis=(0, 1, 2))
    return float(z[above][np.where(col)[0].max()] * 1e6) if col.any() else 0.0

# No per-condition marking here. An earlier version ringed the IN718 49 um spot
# as an out-of-sample spot size, but Ti-6Al-4V spans three spot sizes (47.5, 60
# and 130 um) unmarked, so singling out one alloy's was inconsistent as well as
# one more key in an already crowded legend. The spot-size point is made in the
# text instead.
sum_rows, dropped = [], []
for name, anchor, series, cases in FUSION:
    pts = amb718_targets() if anchor.startswith('amb718') else anchor_points(anchor, series=series)
    for case, P, V, r0 in cases:
        d = load(case, GEOM_BEAM)
        if not mm.domain_validity(d).ok:
            dropped.append(f'{case} (domain)'); continue
        g = mm.fusion_geometry_auto(d)
        dep = mm.depression(d, laser_off_s=manifest(case)['derived'].get('laser_off_time_s'))
        md, mw = target(pts, P, V, r0)
        if not np.isfinite(md):
            dropped.append(f'{case} (no target)'); continue
        sum_rows.append(dict(mat=name, case=case, kind='fusion', P=P, V=V,
                             sim_d=g.depth_um, meas_d=md, sim_w=g.width_um, meas_w=mw,
                             crown=crown_um(d), cav_aspect=dep.aspect))
for case, P, V in DEPRESSION:
    d = load(case, GEOM_BEAM)
    dep = mm.depression(d, laser_off_s=manifest(case)['derived'].get('laser_off_time_s'))
    md = np.nan
    for q in anchor_points('gan_points.json'):
        if abs(q['power_w'] - P) < 2. and abs(q['speed_m_s'] - V) < 1e-6:
            md = float(q['depth_um']); break
    if not np.isfinite(md):
        dropped.append(f'{case} (no Gan target)'); continue
    sum_rows.append(dict(mat='Ti-6Al-4V', case=case, kind='depression', P=P, V=V,
                         sim_d=dep.depth_um, meas_d=md, sim_w=np.nan, meas_w=np.nan,
                         crown=0.0, cav_aspect=dep.aspect))
print('dropped:', dropped or 'none')
nf = sum(1 for r in sum_rows if r['kind'] == 'fusion')
nd = sum(1 for r in sum_rows if r['kind'] == 'depression')
print(f'{len(sum_rows)} conditions: {nf} fusion (depth+width), {nd} depression')
for m in MARK:
    g = [r for r in sum_rows if r['mat'] == m]
    if g:
        e = [100*(r['sim_d']-r['meas_d'])/r['meas_d'] for r in g]
        print(f"  {m:10s} n={len(g):2d}  mean depth error {np.mean(e):+6.1f} %")


In [ ]:
# ── the two summary panels ────────────────────────────────────────────────────
import matplotlib.patches as mpatches
FUS = [r for r in sum_rows if r['kind'] == 'fusion']
DEP = [r for r in sum_rows if r['kind'] == 'depression']
TIC = MCOL['Ti-6Al-4V']

# (a) parity. Filled = fusion depth, open = fusion width, crosses = depression
# depth (Ti-6Al-4V only, so drawn in the Ti-6Al-4V colour). The crown-referenced
# bars that used to hang off each fusion-depth point were removed 2026-09-21
# (user request); the crown height is still computed in sum_rows for the text.
fig, ax = figstyle.panel(frac=0.49, aspect=0.92)
lo, hi = 20., 620.
grid = np.array([lo, hi])
ax.fill_between(grid, grid * (1 - BAND), grid * (1 + BAND), color='0.6',
                alpha=0.18, lw=0)
ax.plot(grid, grid, 'k-', lw=1.0)
for r in FUS:
    c, mk = MCOL[r['mat']], MARK[r['mat']]
    ax.plot(r['meas_d'], r['sim_d'], mk, color=c, ms=6.5, zorder=3)
    if np.isfinite(r['meas_w']):
        ax.plot(r['meas_w'], r['sim_w'], mk, color=c, ms=6.5, mfc='none', zorder=3)
for r in DEP:
    ax.plot(r['meas_d'], r['sim_d'], '+', color=TIC, ms=8, mew=1.8, zorder=4)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel(r'Measured length ($\mu$m)')
ax.set_ylabel(r'Simulated length ($\mu$m)')
key = [plt.Line2D([], [], ls='', marker=MARK[m], color=MCOL[m], ms=6.5, label=m)
       for m in MARK]
key += [plt.Line2D([], [], ls='', marker='o', color='0.35', ms=6.5, label='fusion depth'),
        plt.Line2D([], [], ls='', marker='o', color='0.35', ms=6.5, mfc='none',
                   label='fusion width'),
        plt.Line2D([], [], ls='', marker='+', color=TIC, ms=8, mew=1.8,
                   label='depression depth')]
# Above the axes in four columns: seven keys at a legible size do not fit inside
# a panel this dense; four columns give two rows.
ax.legend(handles=key, frameon=False, fontsize=10, loc='lower center',
          bbox_to_anchor=(0.5, 1.01), ncol=4, handletextpad=.4,
          columnspacing=.9, labelspacing=.35)
figstyle.save(fig, 'calib_summary_parity')

# (b) the same points against the shape of the cavity that produced them. This is
# the panel that shows WHERE the solver is accurate: the deficit is a
# shallow-cavity phenomenon and it closes as the cavity becomes a keyhole.
fig, ax = figstyle.panel(frac=0.49, aspect=0.92)
ax.axhspan(-BAND * 100, BAND * 100, color=figstyle.PALETTE[0], alpha=.13, lw=0)
ax.axhline(0, color='k', lw=.6)
for r in FUS:
    err = 100 * (r['sim_d'] - r['meas_d']) / r['meas_d']
    ax.plot(r['cav_aspect'], err,
            MARK[r['mat']], color=MCOL[r['mat']], ms=6.5, zorder=3)
for r in DEP:
    ax.plot(r['cav_aspect'], 100 * (r['sim_d'] - r['meas_d']) / r['meas_d'], '+',
            color=TIC, ms=8, mew=1.8, zorder=4)
ax.set_xlabel('Simulated cavity aspect ratio')
ax.set_ylabel('Depth error (%)')   # depths only in this panel, so no ambiguity
ax.set_xlim(0, 1.75)
# Two arrows, not a dividing line. Conduction gives way to keyholing gradually
# and nothing in this data marks the crossing, so a line at aspect = 1 asserted
# a threshold we cannot place. The arrows say which way each regime lies and
# claim nothing about where it starts.
y0, y1 = ax.get_ylim()
ax.set_ylim(y0, y1 + 0.16 * (y1 - y0))
ya = ax.get_ylim()[1] - 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0])
_arr = dict(arrowstyle='-|>', color='0.45', lw=1.1, shrinkA=2, shrinkB=0)
_sty = dict(fontsize=11, color='0.45', va='center', annotation_clip=False,
            arrowprops=_arr)
ax.annotate('conduction', xy=(0.02, ya), xytext=(0.87, ya), ha='right', **_sty)
ax.annotate('deep keyhole', xy=(1.73, ya), xytext=(0.93, ya), ha='left', **_sty)
figstyle.save(fig, 'calib_summary_regime')